# ENet compression sweep report

Single reporting point for the whole experiment (`agent_instructions_1.yaml` /
`enet_finn_compression_plan_1.md`). One section per stage; each reads
`compression/results.csv` filtered by `stage` and renders that stage's table
+ plot. Re-run end-to-end after each stage's runs land -- this notebook is
how checkpoints get reported, not ad hoc scripts.

Stage 3 was merged into Stage 1b (structural decisions locked together
before the Stage 2 capacity grid); Stage 5 (FINN folding) is out of scope
for now, so those sections are omitted.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[1] if (Path.cwd().name == "notebook") else Path.cwd()
COMPRESSION_DIR = REPO_ROOT / "compression"
RESULTS_CSV = COMPRESSION_DIR / "results.csv"
COST_TABLES_DIR = COMPRESSION_DIR / "cost_tables"


def load_all() -> pd.DataFrame:
    if not RESULTS_CSV.exists():
        print(f"{RESULTS_CSV} does not exist yet -- no runs collected. Run collect_results.py first.")
        return pd.DataFrame()
    return pd.read_csv(RESULTS_CSV)


def load_stage(stage: str) -> pd.DataFrame:
    df = load_all()
    if df.empty:
        return df
    stage_df = df[df["stage"] == stage].copy()
    if stage_df.empty:
        print(f"No rows yet for stage={stage!r}.")
    return stage_df


def show_table(df: pd.DataFrame, columns: list[str] | None = None) -> None:
    if df.empty:
        return
    display(df[columns] if columns else df)


def plot_dice_vs(df: pd.DataFrame, x_col: str, title: str, label_col: str = "config_name") -> None:
    if df.empty or x_col not in df.columns or "dice" not in df.columns:
        return
    plotted = df.dropna(subset=[x_col, "dice"])
    if plotted.empty:
        print(f"Nothing to plot for {title} (missing {x_col} or dice values).")
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(plotted[x_col], plotted["dice"])
    for _, row in plotted.iterrows():
        ax.annotate(str(row[label_col]), (row[x_col], row["dice"]), fontsize=8,
                    textcoords="offset points", xytext=(4, 4))
    ax.set_xlabel(x_col)
    ax.set_ylabel("Dice")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()

## Stage 1 -- baselines

`enet_paper` vs `E1` (tuned). Sets `D0`/`F0`/`C0` (best Dice/FLOPs/clDice)
that every later stage's goals are defined against.

In [ ]:
stage1 = load_stage("stage1")
show_table(stage1, ["config_name", "f_i", "f1", "f2", "f3", "f4", "f5",
                     "decoder_type", "params", "flops", "dice", "cldice", "n_components"])
plot_dice_vs(stage1, "params", "Stage 1: Dice vs. Params")

if not stage1.empty:
    best = stage1.loc[stage1["dice"].idxmax()]
    D0, F0, C0 = best["dice"], best["flops"], best["cldice"]
    print(f"D0={D0:.4f} (dice) F0={F0:.0f} (flops) C0={C0:.4f} (cldice) -- from {best['config_name']}")
    print(f"Stage-1 goals: dice >= {0.9*D0:.4f}, params < 75000, flops <= {0.3*F0:.0f}")

## Stage 1b -- structure lock (decoder + op-type ablation, merged with the former Stage 3)

Decoder-topology pick (`e1_unpool` vs `e1_upsample`) and op-type ablation
(`no_dilated`/`no_asymmetric`/`no_strided`), both built directly off the
Stage-1 `E1` baseline. Output: `Cfinal_ops`.

In [ ]:
stage1b = load_stage("stage1b")
show_table(stage1b, ["config_name", "decoder_type", "ops_flags", "params", "flops", "dice", "cldice", "n_components"])
plot_dice_vs(stage1b, "params", "Stage 1b: Dice vs. Params (decoder + op ablation)")

## Early probe P2 -- quantization scouting

Decoupled from the main path: runs any time after Foundation is done, on E1
or U4, bits [32, 8, 4, 2]. Scouting only -- Stage 4 still runs properly on
the final architecture. (P1 FINN-resource probe: skipped, FINN out of scope
for now.)

In [ ]:
early_probe_p2 = load_stage("early_probe_p2")
show_table(early_probe_p2, ["config_name", "quant_bits", "params", "bops", "dice", "cldice", "n_components"])
plot_dice_vs(early_probe_p2, "bops", "P2: Dice vs. BOPs")

## Stage 2 -- architecture grid (filters x bottlenecks, on `Cfinal_ops`)

In [ ]:
filter_cost_path = COST_TABLES_DIR / "filter_cost.csv"
bottleneck_cost_path = COST_TABLES_DIR / "bottleneck_cost.csv"
if filter_cost_path.exists():
    display(pd.read_csv(filter_cost_path))
else:
    print(f"{filter_cost_path} not generated yet.")
if bottleneck_cost_path.exists():
    display(pd.read_csv(bottleneck_cost_path))
else:
    print(f"{bottleneck_cost_path} not generated yet.")

In [ ]:
stage2_grid = load_stage("stage2")
show_table(stage2_grid, ["config_name", "f_i", "f1", "f2", "f3", "f4", "f5",
                          "bottlenecks_per_stage", "params", "flops", "dice", "cldice", "n_components"])
plot_dice_vs(stage2_grid, "params", "Stage 2 grid: Dice vs. Params")
plot_dice_vs(stage2_grid, "flops", "Stage 2 grid: Dice vs. FLOPs")

## Stage 2.4 -- f_i reduction (on the chosen grid point)

In [ ]:
stage2_4 = load_stage("stage2_4")
show_table(stage2_4, ["config_name", "f_i", "params", "flops", "dice", "cldice"])
plot_dice_vs(stage2_4, "f_i", "Stage 2.4: Dice vs. f_i")

## Stage 4 -- quantization (Brevitas QAT, on `Cfinal_arch`)

Homogeneous + heterogeneous sweeps. QONNX export step skipped (FINN out of
scope for now) -- this stops at selecting the Pareto-optimal quantized
config.

In [ ]:
stage4 = load_stage("stage4")
show_table(stage4, ["config_name", "quant_bits", "params", "bops", "dice", "cldice", "n_components"])
plot_dice_vs(stage4, "bops", "Stage 4: Dice vs. BOPs")